In [1]:
# ============================================================
# CELL 1: Setup Environment and Configuration
# ============================================================
# FIX (H10 - frontier-vs-legacy generation parity): upgraded the Mistral-family
# generator from Ministral-8B-Instruct-2410 (Oct 2024) to the current generation
# of the SAME 8B-dense line, Ministral-3-8B-Instruct-2512 (Dec 2025). This keeps
# the model size/footprint comparable to before (still runnable in 4-bit) while
# making it a comparable *generation* to the frontier GPT generator.
#   WARNING: Ministral 3 uses Mistral's newer model class / tokenizer backend and
#   is multimodal (text+vision); we only use text. Verify the load + a test
#   generation on your GPU before running the full sweep.
!pip install -q -U "transformers>=4.50" accelerate bitsandbytes "mistral-common>=1.5"

import torch
from transformers import Mistral3ForConditionalGeneration, AutoTokenizer, BitsAndBytesConfig

# Current-generation Mistral model (same Ministral 8B-dense line, Dec 2025).
MODEL_ID = "mistralai/Ministral-3-8B-Instruct-2512"

# Securely fetch HF_TOKEN from Colab Secrets
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if not HF_TOKEN:
        print("⚠️ Warning: HF_TOKEN not found in Colab Secrets.")
except Exception:
    HF_TOKEN = None
    print("⚠️ Warning: Could not access Colab Secrets. Ensure the model is public or token is provided.")

print(f"✅ Ready to connect to: {MODEL_ID}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 144.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 138.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 141.8 MB/s eta 0:00:00
✅ Ready to connect to: mistralai/Ministral-3-8B-Instruct-2512


In [2]:
# ============================================================
# CELL 2: Lightweight Model Verification
# ============================================================

if 'model' in locals() and model is not None:
    print("✅ Model already loaded in memory. Skipping download to save time/VRAM.")
else:
    print("🚀 Model not found. Loading Ministral-3-8B now...")

    # 1. Removed the incompatible bnb_config definition

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 2. Modified from_pretrained to remove the quantization_config parameter
    model = Mistral3ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        device_map="auto",
        token=HF_TOKEN,
        trust_remote_code=True,
        attn_implementation="sdpa",
        torch_dtype="auto"  # Recommended to set as "auto" to automatically match the model's native precision
    )
    model.eval()
    print("✅ Ministral-3-8B loaded successfully!")

🚀 Model not found. Loading Ministral-3-8B now...


config.json:   0%|          | 0.00/1.90k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/21.2k [00:00<?, ?B/s]

tekken.json: reconstructing file:   0%|          |  0.00B / 16.8MB            

tekken.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

✅ Ministral-3-8B loaded successfully!


In [3]:
# ============================================================
# CELL 3: Inference Test (Fixed)
# ============================================================
print("\nRunning test generation...")

test_prompt = "Explain why the sky is blue in one sentence."
messages = [{"role": "user", "content": test_prompt}]

# 1. Apply the template
# FIX (H10): return_dict=True yields a BatchEncoding ({'input_ids': ..., ...}) so
# the **inputs unpacking below works with the Mistral 3 tokenizer backend.
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to("cuda")

# 2. Generate
with torch.no_grad():
    # 🔥 THE FIX IS HERE: Add '**' before inputs
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

# 3. Decode
decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nResponse: {decoded}")



Running test generation...


[transformers] Both `max_new_tokens` (=50) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Response: Explain why the sky is blue in one sentence.The sky appears blue because Earth's atmosphere scatters shorter blue wavelengths of sunlight more than other colors due to **Rayleigh scattering**.


In [ ]:
# ============================================================
# CELL 3: Load your arXiv CSV and preview it
# ============================================================

import pandas as pd
import io
SAMPLE_CSV = """title,abstract,html_url
"Attention Is All You Need","We propose the Transformer, a model based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.","https://ar5iv.labs.arxiv.org/html/1706.03762"
"BERT: Pre-training of Deep Bidirectional Transformers","We introduce BERT, a new language representation model designed to pre-train deep bidirectional representations from unlabeled text.","https://ar5iv.labs.arxiv.org/html/1810.04805"
"""
df = pd.read_csv("ST-Bench_all_domain.csv")
CSV_PATH = "sample_synthetic"   # just a label for option C

# ── Uncomment if using Option A or B and comment out Option C above ──
# df = pd.read_csv(CSV_PATH)

# ------------------------------------------------------------------
# Validate expected columns
# ------------------------------------------------------------------
REQUIRED_COLS = {"title", "abstract"}
missing = REQUIRED_COLS - set(df.columns)
if missing:
    raise ValueError(
        f"CSV is missing required columns: {missing}\n"
        f"Found columns: {list(df.columns)}"
    )

# Optional columns
HAS_HTML_URL   = "html_url"   in df.columns
HAS_FULL_TEXT  = "full_text"  in df.columns

print(f"✅ Loaded CSV: {CSV_PATH}")
print(f"   Rows       : {len(df)}")
print(f"   Columns    : {list(df.columns)}")
print(f"   html_url   : {'✅ present' if HAS_HTML_URL  else '❌ absent (will use abstract only)'}")
print(f"   full_text  : {'✅ present' if HAS_FULL_TEXT else '❌ absent'}")
print()

# Preview
df.head(3)


✅ Loaded CSV: sample_synthetic
   Rows       : 150
   Columns    : ['title', 'abstract', 'license', 'url', 'html_url', 'date']
   html_url   : ✅ present
   full_text  : ❌ absent



,title,abstract,license,url,html_url,date
0,MediX-R1: Open Ended Medical Reinforcement Lea...,"We introduce MediX-R1, an open-ended Reinforce...",http://creativecommons.org/licenses/by-nc-sa/4.0/,https://arxiv.org/abs/2602.23363,https://arxiv.org/html/2602.23363,2026-02-26
1,Model Agreement via Anchoring,Numerous lines of aim to control $\textit{mode...,http://creativecommons.org/licenses/by/4.0/,https://arxiv.org/abs/2602.23360,https://arxiv.org/html/2602.23360,2026-02-26
2,A Dataset is Worth 1 MB,A dataset server must often distribute the sam...,http://creativecommons.org/licenses/by/4.0/,https://arxiv.org/abs/2602.23358,https://arxiv.org/html/2602.23358,2026-02-26


In [5]:
# --- pipeline bootstrap: single source of truth is pipeline.py ---
# Makes pipeline.py importable whether running from a local repo checkout or in
# Google Colab, then imports the shared article fetch/clean helpers. Edit the
# fetch/clean logic ONCE in pipeline.py - not here, and not per-notebook.
import os, sys

def _ensure_pipeline_importable():
    try:
        import pipeline  # noqa: F401
        return
    except ImportError:
        pass
    # Local checkout: walk up from the CWD looking for pipeline.py
    here = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(here, "pipeline.py")):
            sys.path.insert(0, here)
            return
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    # Colab / fresh runtime: fetch pipeline.py from the repo's main branch
    import urllib.request
    url = "https://raw.githubusercontent.com/Dorothy99-love/Style-transfer/main/pipeline.py"
    urllib.request.urlretrieve(url, "pipeline.py")
    sys.path.insert(0, os.getcwd())

_ensure_pipeline_importable()
from pipeline import fetch_html_body_content, get_article_snippet_without_abstract


In [6]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

In [7]:
import traceback
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
# from pipeline import fetch_html_body_content

# Initialize columns for the results
df["mixtral_response"] = ""  # Ensure the target column exists

In [8]:
#Load files in
import time

# 1. Load your specific input file
INPUT_FILE = "ST-Bench_all_domain.csv"
df = pd.read_csv(INPUT_FILE)

# 2. Set the output column name (kept as 'mixtral_response' to match your snippet)
OUTPUT_COL = "mixtral_response"
df[OUTPUT_COL] = ""

# 3. Use T4/A100 device detection
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"✅ Loaded {len(df)} rows. Results will be saved to column: '{OUTPUT_COL}'")

✅ Loaded 150 rows. Results will be saved to column: 'mixtral_response'


In [ ]:
# --- Main evaluation loop for Ministral-3-8B ---
print(f"Starting batch processing of {len(df)} papers...")
print("=" * 50)

# --- Main evaluation loop for Ministral-8B ---
from transfer_prompt import transfer_prompt
for index, row in df.iterrows():
    try:
        html_url = row.get("html_url", None)
        if not html_url or str(html_url) == "nan":
            continue

        # 1. Fetch the content (using your Cell 4 functions)
        article, status = fetch_html_body_content(html_url)
        if status != "html_success":
            print(f"Row {index}: HTML fetch failed ({status})")
            continue

        # 2. Get a larger snippet for a "Full" read
        # 10,000 chars is roughly 2,000-2,500 tokens
        article_snippet = get_article_snippet_without_abstract(article)
        prompt=transfer_prompt+"\n"+"Paper Content\n"+article_snippet
        # 3. Build chat message structure
        messages = [
            {"role": "user", "content": prompt}
        ]

        # 4. Tokenize with Ministral specifics
        # FIX (H10): return_dict=True -> BatchEncoding so **encoded and
        # encoded["input_ids"] below work with the Mistral 3 tokenizer backend.
        encoded = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True,
            truncation=True,
            return_dict=True,
            max_length=131072
        ).to(device)

        # 5. Generate with unpacked dictionary
        with torch.no_grad():
            outputs = model.generate(
                **encoded,
                max_new_tokens=3000,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )

        # 6. Extract generated text
        input_length = encoded["input_ids"].shape[-1]
        generated_tokens = outputs[0][input_length:]

        output_text = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        # 7. Update DataFrame
        df.at[index, OUTPUT_COL] = output_text
        print(f"✅ Row {index}: Successfully processed")
        print("input:",messages)
        print("output:",output_text)

        # Optional: Save progress every 5 rows in case of disconnect
        if (index + 1) % 5 == 0:
            df.to_csv("results_mixtral_all.csv", index=False)

    except Exception as e:
        print(f"❌ Error processing Row {index}: {e}")
        # traceback.print_exc()
        continue

print("\n" + "=" * 50)
print("Batch processing complete.")
# Save the final results to CSV
OUTPUT_FILE = "results_mixtral_all.csv"
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"🎉 Process Finished! Saved {len(df)} rows to {OUTPUT_FILE}")

Starting batch processing of 150 papers...


[transformers] Both `max_new_tokens` (=3000) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Row 0: Successfully processed
input: [{'role': 'user', 'content': '\nYou are a professional science popularization writer.\n\nTask:\nRewrite the following scientific paper content into a clear and accessible popular-science article explanation, targeting the high school students. \n\nRequirements:\n\nYou should avoid equations and heavy jargons and the total length should be 150-200 words. Please consider the following three dimensions when responding. \n1.Style transfer intensity: the scientific content should been effectively transformed into a more accessible, popular-science style which referring to well-known popular science magazines such as "WIRED" or "National Geographic".\n2.Content preservation: the summary should reflect the original article, including all major claims, methodologies, findings and contributions.\n3.Language Naturalness: the summary should read like human-written text.\n\n\nPaper Content\nIntroduction\nLarge medical language and vision-language models are i

[transformers] Both `max_new_tokens` (=3000) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Row 1: Successfully processed
input: [{'role': 'user', 'content': '\nYou are a professional science popularization writer.\n\nTask:\nRewrite the following scientific paper content into a clear and accessible popular-science article explanation, targeting the high school students. \n\nRequirements:\n\nYou should avoid equations and heavy jargons and the total length should be 150-200 words. Please consider the following three dimensions when responding. \n1.Style transfer intensity: the scientific content should been effectively transformed into a more accessible, popular-science style which referring to well-known popular science magazines such as "WIRED" or "National Geographic".\n2.Content preservation: the summary should reflect the original article, including all major claims, methodologies, findings and contributions.\n3.Language Naturalness: the summary should read like human-written text.\n\n\nPaper Content\nIntroduction\nTwo predictive models\nf\n,\nf\n:\n𝒳\n→\nℝ\nf_{1},f_{2}

[transformers] Both `max_new_tokens` (=3000) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Row 2: Successfully processed
input: [{'role': 'user', 'content': '\nYou are a professional science popularization writer.\n\nTask:\nRewrite the following scientific paper content into a clear and accessible popular-science article explanation, targeting the high school students. \n\nRequirements:\n\nYou should avoid equations and heavy jargons and the total length should be 150-200 words. Please consider the following three dimensions when responding. \n1.Style transfer intensity: the scientific content should been effectively transformed into a more accessible, popular-science style which referring to well-known popular science magazines such as "WIRED" or "National Geographic".\n2.Content preservation: the summary should reflect the original article, including all major claims, methodologies, findings and contributions.\n3.Language Naturalness: the summary should read like human-written text.\n\n\nPaper Content\nIntroduction\nSending training datasets from a central server to mult